In [1]:
!pip install pypdf

### Extracte text from pdf and create metadata

In [2]:
from pypdf import PdfReader
import os
#this is the function we we will extracte the text from pdfs:
def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)

    full_text = ""
    for page in reader.pages:
        text = page.extract_text()
        if text:
            full_text += text + "\n"

    return full_text

In [3]:
from langchain_core.documents import Document
"""this function to reade and create metadat from all the files in a folder and transfore
 them to a LC document object"""
def process_folder(folder_path):
    all_documents = []
    folder_name = os.path.basename(os.path.normpath(folder_path))
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            full_path = os.path.join(folder_path, filename)

            try:
                text = extract_text_from_pdf(full_path)
                if not text.strip():
                    print(f"Skipped (empty): {filename}")
                    continue
                  #to not change the name of drugs form A to a i will use replace 2 time
                drug_name = filename.replace(".pdf", "").replace(".PDF", "")

                # document = {
                #     "page_content": text,
                #     "metadata": {
                #         "source": filename,
                #         "drug_name": drug_name,
                #         "file_type": folder_name
                #     }
                # }
                document = Document(
                page_content=text,
                metadata={
                    "source": filename,
                    "drug_name": drug_name,
                    "file_type": folder_name
                    }
            )

                all_documents.append(document)

            except Exception as e:
                print(f"Error reading {filename}: {e}")

    return all_documents

In [4]:
folder_path="/content/dataset/Monoclonal_antibodies"
documents=process_folder(folder_path)

In [44]:
print(os.listdir(folder_path))

['ACTEMRA_TOCILIZUMAB.pdf', 'ABRILADA_ADALIMUMAB-AFZB.pdf', '.ipynb_checkpoints', 'ADAKVEO_CRIZANLIZUMAB-TMCA.pdf', 'ADBRY_TRALOKINUMAB.pdf']


In [62]:
len(documents)

4

In [63]:
type(documents[0])

langchain_core.documents.base.Document

### Create chunks from the extracted text with overlaping


In [34]:
!pip install langchain-text-splitters

In [5]:
import langchain_core
import langchain
from langchain_text_splitters import RecursiveCharacterTextSplitter

for now I will suse the chunk size to 1000 and overlaping 200 and that is about 150-200 words

In [6]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""]  # Priority order
    )
    chunks = splitter.split_documents(documents)
    return chunks

In [7]:
chunks = split_documents(documents)
len(chunks)

649

### Embedding

In [68]:
!pip install langchain-google-genai langchain-pinecone pinecone-client

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.4/127.4 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.3 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.3
    Uninstalling packaging-26.3:
      Successfully uninstalled packaging-26.3
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.

In [8]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [9]:
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("GEM_API_KEY")

In [29]:
gemini_embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    output_dimensionality=768
)

In [12]:
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone

os.environ["PINECONE_API_KEY"] = userdata.get("PINECONE_API_KEY")
pc = Pinecone()

creating an index for the pino database

In [30]:
from pinecone import ServerlessSpec

index_name = "fda-drug-labels"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=768,   #demention of the gemini embedding model
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f"the index is created")
else:
    print(f"the index is already exists: {index_name}")


the index is already exists: fda-drug-labels


In [21]:
index = pc.Index(index_name)

In [31]:
vectorstore = PineconeVectorStore(
    index=index,
    embedding=gemini_embeddings
)

In [26]:
vectorstore.add_documents(chunks)


GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-1.0\nPlease retry in 20.704994713s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerMinutePerUserPerProjectPerModel-FreeTier', 

so because we have limits in the free embedding model form gemini we need to controle the speed of the requeste sending

In [32]:
import time

def upload_chunks_with_throttle(vectorstore, chunks, batch_size=50, delay_seconds=61):
    total = len(chunks)
    for i in range(0, total, batch_size):
        batch = chunks[i:i + batch_size]
        batch_num = (i // batch_size) + 1
        total_batches = (total + batch_size - 1) // batch_size

        print(f"  Batch {batch_num}/{total_batches} ({len(batch)} chunks)")

        try:
            vectorstore.add_documents(batch)
            print(f"Success")
        except Exception as e:
            print(f" Error: {e}")
            print(f"    Waiting 65s and retrying")
            time.sleep(65)
            vectorstore.add_documents(batch)
            print(f"Retry successful")

        # Wait before the next batch (skip wait after the last batch)
        if i + batch_size < total:
            print(f"Waiting before next batch")
            time.sleep(delay_seconds)

    print(f"\nDone")

In [33]:
upload_chunks_with_throttle(
    vectorstore=vectorstore,
    chunks=chunks
)

  Batch 1/13 (50 chunks)
Success
Waiting before next batch
  Batch 2/13 (50 chunks)
Success
Waiting before next batch
  Batch 3/13 (50 chunks)
Success
Waiting before next batch
  Batch 4/13 (50 chunks)
Success
Waiting before next batch
  Batch 5/13 (50 chunks)
Success
Waiting before next batch
  Batch 6/13 (50 chunks)
Success
Waiting before next batch
  Batch 7/13 (50 chunks)
Success
Waiting before next batch
  Batch 8/13 (50 chunks)
Success
Waiting before next batch
  Batch 9/13 (50 chunks)
Success
Waiting before next batch
  Batch 10/13 (50 chunks)
Success
Waiting before next batch
  Batch 11/13 (50 chunks)
Success
Waiting before next batch
  Batch 12/13 (50 chunks)
Success
Waiting before next batch
  Batch 13/13 (49 chunks)
Success

Done


In [34]:
query = "What drug treats Crohn's disease?"
results = vectorstore.similarity_search(query, k=3)

In [37]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(f"Drug: {doc.metadata['drug_name']}")
    print(f"Source: {doc.metadata['source']}")
    print(f"Text: {doc.page_content[:500]}...")


--- Result 1 ---
Drug: ABRILADA_ADALIMUMAB-AFZB
Source: ABRILADA_ADALIMUMAB-AFZB.pdf
Text: Crohn’s disease (CD) is based on body weight as shown below:
Pediatric Weight
Recommended Dosage
Days 1 through 15 Starting on Day 29
17 kg (37 lbs) to less than 
40 kg (88 lbs)
Day 1: 80 mg 
Day 15: 40 mg 
20 mg every other week 
40 kg (88 lbs) and greater Day 1: 160 mg (single dose or split over two 
consecutive days) 
Day 15: 80 mg 
40 mg every other week 
This label may not be the latest approved by FDA.  
For current labeling information, please visit https://www.fda.gov/drugsatfda
2.4 Ulce...

--- Result 2 ---
Drug: ABRILADA_ADALIMUMAB-AFZB
Source: ABRILADA_ADALIMUMAB-AFZB.pdf
Text: 15 kg (33 lbs) to less than 30 kg (66 lbs) 20 mg every other week 
30 kg (66 lbs) and greater 40 mg every other week 
Crohn's Disease (2.3):
 Adults: 160 mg on Day 1 (given in one day or split over two consecutive 
days); 80mg on Day 15; and 40 mg every other week starting on Day 29.
 Pediatric Patients 6 Yea